# Barebones SpatialData Image Write Test (`sopa.io.ome_tif`, 10k crop)

This notebook is the Sopa-reader counterpart to the manual barebones test.

It does only four things:

1. inspect the merged OME-TIFF metadata
2. lazily read the full image with `sopa.io.ome_tif(..., as_image=True)`
3. crop a 10k x 10k window
4. parse it back into `Image2DModel`
5. write the image-only SpatialData store
6. reopen the store and inspect it

No labels, shapes, segmentation, or tables are included.


In [8]:
from __future__ import annotations

import os
import json
import shutil
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import sopa.io
import spatialdata as sd
import tifffile
from spatialdata import SpatialData
from spatialdata.models import Image2DModel


In [9]:
OUTPUTS_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs")
FULL_MERGE_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif")
CHANNEL_MAP_PATH = OUTPUTS_DIR / "channel_map.generated.json"
IMAGE_LAYER = "full_image"
CHUNK_SHAPE = (1, 512, 512)
CROP_X = 0
CROP_Y = 0
CROP_WIDTH = 2000
CROP_HEIGHT = 2000
SCALE_FACTORS = None #[2, 2, 2, 2]
WRITE_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_sopa_ome_tif_SLIDE-0329_crop_2000.sdata.zarr")

assert FULL_MERGE_PATH.exists(), FULL_MERGE_PATH
assert CHANNEL_MAP_PATH.exists(), CHANNEL_MAP_PATH

print({
    "spatialdata": sd.__version__,
    "full_merge_path": str(FULL_MERGE_PATH),
    "write_path": str(WRITE_PATH),
    "chunk_shape": CHUNK_SHAPE,
    "crop_box": {"x": CROP_X, "y": CROP_Y, "width": CROP_WIDTH, "height": CROP_HEIGHT},
    "scale_factors": SCALE_FACTORS,
})


{'spatialdata': '0.7.2', 'full_merge_path': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif', 'write_path': '/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_sopa_ome_tif_SLIDE-0329_crop_2000.sdata.zarr', 'chunk_shape': (1, 512, 512), 'crop_box': {'x': 0, 'y': 0, 'width': 2000, 'height': 2000}, 'scale_factors': None}


In [10]:
channel_map = json.loads(CHANNEL_MAP_PATH.read_text())
channel_aliases = [entry["alias"] for entry in channel_map]

with tifffile.TiffFile(FULL_MERGE_PATH) as tif:
    series = tif.series[0]
    series_shape = tuple(int(value) for value in series.shape)
    series_axes = series.axes
    level_shapes = [tuple(int(value) for value in level.shape[-2:]) for level in series.levels]
    level0_dtype = str(series.levels[0].dtype)
    page0 = series.levels[0].pages[0]
    tile_info = {
        "is_tiled": bool(page0.is_tiled),
        "tilewidth": int(page0.tilewidth) if page0.is_tiled else None,
        "tilelength": int(page0.tilelength) if page0.is_tiled else None,
        "compression": str(page0.compression),
    }

assert CROP_X >= 0 and CROP_Y >= 0, "Crop origin must be non-negative"
assert CROP_X + CROP_WIDTH <= level_shapes[0][1], "Crop width exceeds image bounds"
assert CROP_Y + CROP_HEIGHT <= level_shapes[0][0], "Crop height exceeds image bounds"

print({
    "series_shape": series_shape,
    "series_axes": series_axes,
    "level_count": len(level_shapes),
    "level0_shape": level_shapes[0],
    "level0_dtype": level0_dtype,
    "first_channels_from_map": channel_aliases[:5],
    "tile_info": tile_info,
    "crop_shape_yx": (CROP_HEIGHT, CROP_WIDTH),
})


{'series_shape': (25, 62617, 66406), 'series_axes': 'CYX', 'level_count': 8, 'level0_shape': (62617, 66406), 'level0_dtype': 'uint16', 'first_channels_from_map': ['R1_DAPI_AF', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI'], 'tile_info': {'is_tiled': True, 'tilewidth': 256, 'tilelength': 256, 'compression': '8'}, 'crop_shape_yx': (2000, 2000)}


In [4]:
source_image = sopa.io.ome_tif(FULL_MERGE_PATH, as_image=True)
cropped_image = source_image.isel(y=slice(CROP_Y, CROP_Y + CROP_HEIGHT), x=slice(CROP_X, CROP_X + CROP_WIDTH))
cropped_image = cropped_image.chunk({"c": 1, "y": CHUNK_SHAPE[1], "x": CHUNK_SHAPE[2]})

full_image = Image2DModel.parse(
    cropped_image,
    dims=("c", "y", "x"),
    c_coords=channel_aliases,
    chunks=CHUNK_SHAPE,
    scale_factors=SCALE_FACTORS,
)

sdata = SpatialData(images={IMAGE_LAYER: full_image})
# scale0 = sdata.images[IMAGE_LAYER]["scale0"].image

# print({
#     "source_image_type": type(source_image).__name__,
#     "source_shape": tuple(source_image.shape),
#     "source_chunks": source_image.chunks,
#     "cropped_shape": tuple(cropped_image.shape),
#     "cropped_chunks": cropped_image.chunks,
#     "image_type": type(full_image).__name__,
#     "pyramid_levels": list(full_image.keys()),
#     "scale0_shape": tuple(scale0.shape),
#     "scale0_dims": tuple(scale0.dims),
#     "scale0_chunks": scale0.chunks,
#     "first_channels": scale0.coords["c"].values[:5].tolist(),
# })

sdata


SpatialData object
└── Images
      └── 'full_image': DataArray[cyx] (24, 2000, 2000)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)

In [5]:
if WRITE_PATH.exists():
    shutil.rmtree(WRITE_PATH)

sdata.write(WRITE_PATH, overwrite=True)
print("Wrote:", WRITE_PATH)


Wrote: /mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_sopa_ome_tif_SLIDE-0329_crop_2000.sdata.zarr


In [ ]:
reloaded = sd.read_zarr(WRITE_PATH)
reloaded_scale0 = reloaded.images[IMAGE_LAYER]["scale0"].image

print(reloaded)
print({
    "images": list(reloaded.images.keys()),
    "labels": list(reloaded.labels.keys()),
    "tables": list(reloaded.tables.keys()),
    "scale0_shape": tuple(reloaded_scale0.shape),
    "scale0_chunks": reloaded_scale0.chunks,
})
